John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

In [1]:
library(tidyverse)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'tibble' was built under R version 4.4.3"
Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'readr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'forcats' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package 

In [2]:
rawdata <- read_csv("data/megazord.csv")

New names:
• `` -> `...1`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 12799 Columns: 127
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (14): ID, recordist, Begin.Path, Location_Number, Comments, ST1, Data_...
dbl  (104): ...1, X.2, X.1, selec, start, end, File.Offset..s., SNR.NIST.Qui...
lgl    (2): RepeatST, GreaterThanT
dttm   (1): DateTime
date   (1): date
time   (5): click_clock, start_hms, click_clock_hms, clock_time, sunrise

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
columns_of_interest <- c(
    "ID", # Which male is singing (each has a unique identifier)
    "DateTime", # Date AND Time
    "time_rel_sunrise", # How much time has passed since sunrise
    "UST", # Universal Song Type
    "IST", # Individual Song Type 
    "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6", "D_M2_M3", "D_M2_M4", "D_M2_M5", "D_M2_M6", "D_M3_M4", "D_M3_M5", "D_M3_M6", "D_M4_M5", "D_M4_M6", "D_M5_M6", # IID - inter-individual distance between Males 
    "M1_Lat", "M1_Lon", "M2_Lat", "M2_Lon", "M3_Lat", "M3_Lon", "M4_Lat", "M4_Lon", "M5_Lat", "M5_Lon", "M6_Lat", "M6_Lon", # Latitude and Longitude
    "M1_last_UST", "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST", # Last song type
    "M1_match", "M2_match", "M3_match", "M4_match", "M5_match", "M6_match",  # Does focal male match with Male X's last song? (Cannot match with itself)
    "number_of_matches", # How many males' last song matches with the focal male's song?
    "does_it_match", # Does the focal male's current song match with any other male's most recent song?
    "neighbourhood", # North, Centre, or South. We'll be starting our analysis on the North neighbourhood
    "Location_Number", 
    "Time_Diff", # Time since this bird's own last song
    "RepeatST", # Is this a repeat of a song the bird just sang?
    "Repeat_ST_Run" # How many songs of the same type have been repeated in this run? ("Streak")
    # "Phase" # DC (Dawn Chorus) or PDC (Post-Dawn Chorus) TODO: Filter out PDC data
    # "Data_Class" # We are only interested in "song"
    )

In [4]:


data <- rawdata %>% 
    filter(Data_Class == "song") %>% # Only interested in song data
    filter(Phase == "DC") %>% # Only interested in Dawn Chorus songs
    filter(LOKI == 0) %>% # Remove Kingbird interference
    select(columns_of_interest) %>% 
    arrange(desc("DateTime")) %>%
    mutate(UST = as.factor(UST)) # Song type is categorical attribute, not numerical

summary(data)

Warning message:
"Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(columns_of_interest)

  # Now:
  data %>% select(all_of(columns_of_interest))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>."


      ID               DateTime                      time_rel_sunrise 
 Length:7941        Min.   :2022-05-09 05:19:33.00   Min.   :-2509.1  
 Class :character   1st Qu.:2022-05-14 05:36:24.00   1st Qu.:-1635.6  
 Mode  :character   Median :2022-05-18 05:26:24.00   Median :-1270.8  
                    Mean   :2022-05-17 14:40:09.15   Mean   :-1244.1  
                    3rd Qu.:2022-05-20 05:57:02.00   3rd Qu.: -897.5  
                    Max.   :2022-05-24 06:01:11.00   Max.   : 1125.2  
                                                                      
      UST           IST               D_M1_M2          D_M1_M3      
 9      : 542   Length:7941        Min.   :  0.00   Min.   : 31.95  
 8      : 472   Class :character   1st Qu.: 58.74   1st Qu.: 47.44  
 1      : 466   Mode  :character   Median : 78.37   Median :136.18  
 10     : 453                      Mean   : 87.48   Mean   :117.33  
 20     : 424                      3rd Qu.: 94.04   3rd Qu.:152.90  
 (Other):5573     

### Export Data

In [5]:
write_csv(data, "data/minizord.csv")

In [6]:
# This cell used for easy copy/paste of column names

"...1"
"X.2"
"X.1"
"selec"
"start"
"end"
"date"
"ID"
"recordist"
"Begin.Path"
"File.Offset..s."
"SNR.NIST.Quick..dB."
"Location_Number"
"Comments"
"ST1"
"Data_Class"
"selec.file"
"sound.files"
"click_clock"
"click_rec"
"start_hms"
"click_clock_hms"
"click_rec_hms"
"clock_time"
"sunrise"
"time_rel_sunrise"
"M1_Off"
"M2_Off"
"M3_Off"
"M4_Off"
"M5_Off"
"M6_Off"
"Lat"
"Lon"
"M1_Lat"
"M1_Lon"
"M2_Lat"
"M2_Lon"
"M3_Lat"
"M3_Lon"
"M4_Lat"
"M4_Lon"
"M5_Lat"
"M5_Lon"
"M6_Lat"
"M6_Lon"
"D_M1_M2"
"D_M1_M3"
"D_M1_M4"
"D_M1_M5"
"D_M1_M6"
"D_M2_M3"
"D_M2_M4"
"D_M2_M5"
"D_M2_M6"
"D_M3_M4"
"D_M3_M5"
"D_M3_M6"
"D_M4_M5"
"D_M4_M6"
"D_M5_M6"
"IST"
"X"
"UST"
"M1_last_UST"
"M2_last_UST"
"M3_last_UST"
"M4_last_UST"
"M1_match"
"M2_match"
"M3_match"
"M4_match"
"neighbourhood"
"M5_last_UST"
"M6_last_UST"
"M5_match"
"M6_match"
"number_of_matches"
"does_it_match"
"song_end_time"
"DateTime"
"Bird_Date"
"Time_Diff"
"RepeatST"
"GreaterThanT"
"Repeat_ST_Run"
"Long_Latency_Run"
"meets_lat_thresh"
"meets_rep_thresh"
"meets_either_thresh"
"Phase"
"End_DC"
"index"
"orig_start"
"orig_end"
"rms"
"LOKI"
"note_count"
"song_start"
"song_end"
"song_duration"
"note_duration"
"gap_duration"
"percent_sound"
"min_freq"
"max_freq"
"peak_freq"
"max_note_dur"
"mean_freq"
"last_note_dur"
"freq_BW_subtraction"
"freq_BW_ratio"
"TR_unbiased"
"mean_note_dur"
"mean_gap_dur"
"acoustic_outlier"
"avg_note_dur_nl"
"avg_gap_dur_nl"
"avg_note_gap_ratio_nl"
"avg_note_gap_difference_nl"
"avg_min_freq_nl"
"avg_max_freq_nl"
"avg_note_bw_ratio_nl"
"avg_note_FM_nl"
"avg_silent_jump_ratio_nl"
"avg_silent_gap_FM_nl"
"avg_recovery_time_nl"


[1] "...1"

[1] "X.2"

[1] "X.1"

[1] "selec"

[1] "start"

[1] "end"

... (117 outputs left out)

[1] "avg_note_FM_nl"

[1] "avg_silent_jump_ratio_nl"

[1] "avg_silent_gap_FM_nl"

[1] "avg_recovery_time_nl"